# 01. Trie (Prefix Tree): Fundamentals

A trie stores many **words** in a tree where each edge is a **character**. Words that share a prefix share the same path. It answers **prefix questions** very fast.

**Learning order:** REAL LIFE → STRUCTURE → INSERT / SEARCH / STARTSWITH → COUNTS → AUTOCOMPLETE → DELETE → TRIE VS HASH SET → INTERVIEW THINKING

## 1. Real-life examples

- **Phone keyboard / Google autocomplete:** you type `"ap"` and it suggests `apple`, `apply`, `app`.
- **Spell checker**, **contacts search**, **IP routing tables** (longest prefix match).
- **Dictionary games** (Boggle/Scrabble solvers).

**Recognition clue:** "words with a given prefix", "autocomplete", "search many words in a grid", "longest common prefix", "XOR of two numbers" (binary trie).

## 2. Structure

Each node has:
- `children`: dictionary `char → node`
- `is_end`: `True` if a word **ends** here (needed to tell `"app"` from a prefix of `"apple"`)

Words `app`, `apple`, `apt`:

```
root - a - p - p* - l - e*
             \ - t*
```
(`*` = a word ends here)

**Complexity:** insert, search, startsWith are all **O(L)** where `L` is the word length, independent of how many words are stored.

In [1]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.count = 0                  # how many words pass through this node

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.count += 1
        node.is_end = True

    def _walk(self, s):                 # follow s; return the node or None
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end

    def starts_with(self, prefix):
        return self._walk(prefix) is not None

    def count_prefix(self, prefix):
        node = self._walk(prefix)
        return node.count if node else 0

t = Trie()
t.insert("apple")
print(t.search("apple"), t.search("app"), t.starts_with("app"))   # True False True
t.insert("app")
print(t.search("app"), t.count_prefix("ap"))   # True 2

True False True
True 2


## 3. Autocomplete: all words with a prefix

Walk to the prefix node, then **DFS** the subtree collecting words. Cost: O(L + size of the subtree).

In [2]:
def autocomplete(trie, prefix):
    node = trie._walk(prefix)
    if not node:
        return []
    res = []
    def dfs(node, path):
        if node.is_end:
            res.append("".join(path))
        for ch in sorted(node.children):
            path.append(ch)
            dfs(node.children[ch], path)
            path.pop()
    dfs(node, list(prefix))
    return res

t = Trie()
for w in ["apple", "app", "apply", "apt", "bat"]:
    t.insert(w)
print(autocomplete(t, "ap"))   # ['app', 'apple', 'apply', 'apt']

['app', 'apple', 'apply', 'apt']


## 4. Delete a word

Unmark `is_end`. Then, on the way back, **remove nodes that are no longer needed** (no children and not the end of another word). This keeps the trie small.

In [3]:
def delete(trie, word):
    def helper(node, i):
        if i == len(word):
            if not node.is_end:
                return False
            node.is_end = False
            return not node.children              # True = parent can delete this node
        ch = word[i]
        if ch not in node.children:
            return False
        if helper(node.children[ch], i + 1):
            del node.children[ch]
            return not node.children and not node.is_end
        return False
    helper(trie.root, 0)

t = Trie()
for w in ["app", "apple"]:
    t.insert(w)
delete(t, "apple")
print(t.search("apple"), t.search("app"), t.starts_with("appl"))   # False True False

False True False


## 5. Trie vs HashSet vs sorted list

| Need | Trie | HashSet |
|---|---|---|
| exact word lookup | O(L) | O(L) (hashing the string) |
| **prefix** lookup / count | **O(L)** | O(n · L) (scan all) |
| autocomplete in sorted order | natural (DFS in order) | needs sorting |
| memory | **more** (many nodes) | less |

Use a trie when prefix operations matter. For plain membership, a set is simpler and smaller.

## 6. Interview thinking

**Say:** "I'll store the words in a trie so every prefix query costs O(L), independent of the dictionary size. Insert takes O(L) time and at most O(L) new nodes."

**Common mistakes:** forgetting `is_end` (treating a prefix as a word), returning `True` for `search` on a mere prefix, using a list of 26 for huge alphabets (use a dict), not handling an empty string.

**Memory:** worst case O(total characters). A dict per node is flexible, a `[None] * 26` array is faster for lowercase letters.

## Quick Revision

- Node = `children` dict + `is_end`. Shared prefixes share nodes.
- insert / search / startsWith = O(L).
- `search` needs `is_end`, `startsWith` does not.
- Autocomplete = walk to prefix + DFS.
- Delete: unmark, prune unused nodes upward.